# Snippet from Design-Notes.md


In [ ]:
import numpy as np  
from scipy.linalg import eigh  # For eig-decomp  
def riemannian_sgd(embeddings, pairs, lr=0.01, rank=10):  
    # Init low-rank M: UΛU^T  
    D = embeddings.shape[1]  
    U = np.eye(D)[:, :rank]  # Orthogonal basis  
    Λ = np.ones(rank) * 0.1  # Small positives  
    M = U @ np.diag(Λ) @ U.T  
     
    for epoch in range(100):  
        for (x, y) in pairs:  # Positive pairs  
            # Log-Euclidean mean update  
            logM = np.log(M + 1e-6)  # Stabilize  
            grad = (x - y) @ (x - y).T  # Simple divergence proxy  
            logM -= lr * grad  # Gradient step  
            M = np.exp(logM)  # Exponentiate back to SPD  
             
            # Low-rank project: Truncate eig  
            vals, vecs = eigh(M)  
            top_k = np.argsort(vals)[-rank:]  
            M = vecs[:, top_k] @ np.diag(vals[top_k]) @ vecs[:, top_k].T  
     
    return M  # Your manifold metric, ready to route  
# Toy: 2D embeddings, learn separating metric  
emb = np.random.randn(50, 2)  
pairs = [(emb[i], emb[i+1]) for i in range(49)]  
M_learned = riemannian_sgd(emb, pairs)  
print(f"Rank-constrained M:\n{M_learned}")  
# Output: Positive-def, low-rank—distances now "curved" for task.  
